In [ ]:
import numpy as np
import plotly.graph_objects as go



## Current Model

We currently use an ideal differential-drive kinematic model.

State:

$$
q =
\begin{bmatrix}
x\\
y\\
\theta
\end{bmatrix}
$$

Inputs are the wheel angular velocities:

$$
u =
\begin{bmatrix}
\omega_R\\
\omega_L
\end{bmatrix}
$$

Wheel velocities:

$$
v_R = r\omega_R
$$

$$
v_L = r\omega_L
$$

Robot forward velocity:

$$
v = \frac{v_R+v_L}{2}
$$

Robot angular velocity:

$$
\omega = \dot{\theta}
=
\frac{v_R-v_L}{L}
$$

State equations:

$$
\dot{x}=v\cos\theta
$$

$$
\dot{y}=v\sin\theta
$$

$$
\dot{\theta}=\omega
$$

Equivalently, directly from the wheel angular velocities:

$$
\dot{x}
=
\frac{r}{2}
(\omega_R+\omega_L)
\cos\theta
$$

$$
\dot{y}
=
\frac{r}{2}
(\omega_R+\omega_L)
\sin\theta
$$

$$
\dot{\theta}
=
\frac{r}{L}
(\omega_R-\omega_L)
$$


In [ ]:
class DifferentialDriveRobot:
    def __init__(
            self,
            wheel_radius: float,
            wheel_base: float,
            initial_state: tuple[float, float, float],
            omega_cap = None) -> None:
        self.wheel_radius = wheel_radius
        self.wheel_base = wheel_base

        self.current_state: tuple[float, float, float] = initial_state  # x, y, theta
        self.linear_velocity = 0
        self._omega_cap = omega_cap


    def simulation_step(self, omega_l: float, omega_r: float, dt: float) -> None:
        if self._omega_cap:
            omega_l = max(-self._omega_cap, min(self._omega_cap, omega_l))
            omega_r = max(-self._omega_cap, min(self._omega_cap, omega_r))

        x_dot, y_dot, theta_dot, linear_velocity = self._calculate_state_chage(
            self.current_state,
            omega_l,
            omega_r)

        self._update_state(x_dot, y_dot, theta_dot, dt)
        self.linear_velocity = linear_velocity

        return self.current_state


    def print_state(self) -> None:
        x, y, theta = self.current_state
        print(f"x: {x:.2f}, y: {y:.2f}, theta: {theta:.2f}")



    def _calculate_state_chage(self,
                               current_state: tuple[float, float, float],
                               omega_l: float,
                               omega_r: float) -> tuple[float, float, float]:

        _, _, theta = current_state

        x_dot = (self.wheel_radius / 2) * (omega_r + omega_l) * np.cos(theta)
        y_dot = (self.wheel_radius / 2) * (omega_r + omega_l) * np.sin(theta)
        theta_dot = (self.wheel_radius / self.wheel_base) * (omega_r - omega_l)

        linear_velocity = (omega_l + omega_r) / 2

        return x_dot, y_dot, theta_dot, linear_velocity

    def _update_state(self, x_dot: float, y_dot: float, theta_dot: float, dt: float) -> None:
        self.current_state = (
            self.current_state[0] + x_dot * dt,
            self.current_state[1] + y_dot * dt,
            self.current_state[2] + theta_dot * dt
        )


# Experiment 1: One simmulation step

In [ ]:
r = 0.03
L = 0.15
omera_r = 10
omega_l = 5
initial_state = (0, 0, 0)
dt = 0.1

In [ ]:
exp_01_plant = DifferentialDriveRobot(r, L, initial_state)

In [ ]:
print("Initial state:")
exp_01_plant.print_state()

Initial state:
x: 0.00, y: 0.00, theta: 0.00


In [ ]:
exp_01_plant.simulation_step(omega_l, omera_r, dt)

(np.float64(0.0225), np.float64(0.0), 0.1)

In [ ]:
print("after one step:")
exp_01_plant.print_state()

after one step:
x: 0.02, y: 0.00, theta: 0.10


# Experiment 2: Run many steps

In [ ]:
exp_02_plant = DifferentialDriveRobot(r, L, initial_state)

In [ ]:
state_history = [
    initial_state
]
for i in range(60):
    new_state = exp_02_plant.simulation_step(omega_l, omera_r, dt)
    state_history.append(new_state)

    # exp_02_plant.print_state()

In [ ]:
print(state_history[1])

(np.float64(0.0225), np.float64(0.0), 0.1)


In [ ]:
# Keep plots reasonable by downsampling large series
MAX_PLOT_POINTS = 2000

def _downsample_indices(n: int, max_points: int = MAX_PLOT_POINTS):
    if n <= max_points:
        return None
    return np.linspace(0, n - 1, max_points).astype(int)

def plot_trajectory(state_history: list[tuple[float, float, float]]) -> None:
    x_positions = [pos[0] for pos in state_history]
    y_positions = [pos[1] for pos in state_history]

    # downsample if too many points
    n = len(x_positions)
    idx = _downsample_indices(n)
    if idx is not None:
        x_positions = [x_positions[int(i)] for i in idx]
        y_positions = [y_positions[int(i)] for i in idx]

    fig = go.Figure()

    # Add trajectory line
    fig.add_trace(go.Scatter(
        x=x_positions,
        y=y_positions,
        mode='lines+markers',
        name='Robot Path',
        line={"color": 'blue', "width": 2},
        marker={"size": 4}
    ))

    # Add start point
    fig.add_trace(go.Scatter(
        x=[x_positions[0]],
        y=[y_positions[0]],
        mode='markers',
        name='Start',
        marker={"color": 'green', "size": 12, "symbol": 'circle'}
    ))

    # Add end point
    fig.add_trace(go.Scatter(
        x=[x_positions[-1]],
        y=[y_positions[-1]],
        mode='markers',
        name='End',
        marker={"color": 'red', "size": 12, "symbol": 'square'}
    ))

    fig.update_layout(
        title='Robot 2D Trajectory',
        xaxis_title='X Position (m)',
        yaxis_title='Y Position (m)',
        hovermode='closest',
        showlegend=True
    )

    # Make axes equal (1 unit on x = 1 unit on y visually)
    fig.update_yaxes(scaleanchor="x", scaleratio=1)

    fig.show()


def plot_time_series(
        values: list[float],
        dt: float,
        title: str,
        yaxis_title: str,
        series_name: str = 'Value',
        color: str = 'orange') -> None:
    n = len(values)
    idx = _downsample_indices(n)
    if idx is None:
        times = [i * dt for i in range(n)]
        vals = values
    else:
        orig_times = np.arange(n) * dt
        times = [float(orig_times[int(i)]) for i in idx]
        vals = [values[int(i)] for i in idx]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=times,
        y=vals,
        mode='lines+markers',
        name=series_name,
        line={"color": color, "width": 2},
        marker={"size": 4}
    ))

    fig.update_layout(
        title=title,
        xaxis_title='Time (s)',
        yaxis_title=yaxis_title,
        hovermode='closest',
        showlegend=True
    )

    fig.show()

def plot_errors(errors: list[float], dt: float, title: str = None) -> None:
    if title is None:
        title = 'Control Error Over Time'
    plot_time_series(errors, dt, title, 'Error', series_name='Error', color='orange')


def plot_velocity(velocities: list[float], dt: float) -> None:
    plot_time_series(
        velocities,
        dt,
        'Velocity Command Over Time',
        'Linear Velocity (m/s)',
        series_name='Commanded Velocity',
        color='blue'
    )

def plot_theta(thetas: list[float], dt: float) -> None:
    """Plot robot heading over time. Uses np.unwrap to avoid +/-pi jumps."""
    thetas_arr = np.unwrap(np.array(thetas))
    plot_time_series(thetas_arr.tolist(), dt, 'Heading (Theta) Over Time', 'Theta (rad)', series_name='Theta', color='green')


In [ ]:
# # sadplot_trajectory(state_history)

# Experiment 3: Change wheel speeds

In [ ]:
# straight
omega_l = omega_r = 10
exp_03_straight_plant = DifferentialDriveRobot(r, L, initial_state)

state_history = [ initial_state ]
for i in range(60):
    new_state = exp_03_straight_plant.simulation_step(omega_l, omega_r, dt)
    state_history.append(new_state)

plot_trajectory(state_history)

In [ ]:
# rotate in place
omega_l = 10
omega_r = -10
exp_03_straight_plant = DifferentialDriveRobot(r, L, initial_state)

state_history = [ initial_state ]
for i in range(60):
    new_state = exp_03_straight_plant.simulation_step(omega_l, omega_r, dt)
    state_history.append(new_state)

plot_trajectory(state_history)

In [ ]:
# Asymetric
omega_l = 10
omega_r = 13
exp_03_straight_plant = DifferentialDriveRobot(r, L, initial_state)

state_history = [ initial_state ]
for i in range(60):
    new_state = exp_03_straight_plant.simulation_step(omega_l, omega_r, dt)
    state_history.append(new_state)

plot_trajectory(state_history)

In [ ]:
# random wheel speeds
exp_03_random_plant = DifferentialDriveRobot(r, L, initial_state)

state_history = [initial_state]
for i in range(600):
    # Random omega values between -15 and 15 rad/s
    omega_l = np.random.uniform(-15, 15)
    omega_r = np.random.uniform(-15, 15)

    new_state = exp_03_random_plant.simulation_step(omega_l, omega_r, dt)
    state_history.append(new_state)

plot_trajectory(state_history)


# Experiment 4: Investigate dt

In [ ]:
T = 5 # sec
dts = [0.1, 0.01, 0.001]

In [ ]:
for dt in dts:
    exp_04_random_plant = DifferentialDriveRobot(r, L, initial_state)

    state_history = [initial_state]
    for i in range(int(T/dt)):
        omega_l = 10 + (i*dt)
        omega_r = 5 - i * dt
        # print(omega_l)
        # print(omega_r)

        new_state = exp_04_random_plant.simulation_step(omega_l, omega_r, dt)
        state_history.append(new_state)

    plot_trajectory(state_history)

# P contoller

In [ ]:
p_ctrl_plant = DifferentialDriveRobot(r, L, initial_state)

x_target = 10
dt = 0.01
kp = 1

state_history = [initial_state]
errors = []
v_s = []
for i in range(500):
    x_current, _, _ = p_ctrl_plant.current_state
    e = x_target - x_current
    errors.append(e)
    v_needed = kp * e
    v_s.append(v_needed)

    omega_needed = v_needed / r
    new_state = p_ctrl_plant.simulation_step(omega_needed, omega_needed, dt)

    state_history.append(new_state)

plot_trajectory(state_history)
plot_errors(errors, dt)
plot_velocity(v_s, dt)


# Phisical limitation of the system

In [ ]:
x_target = 10
dt = 0.01
kp = 1
p_ctrl_plant = DifferentialDriveRobot(r, L, initial_state, omega_cap = 20)

state_history = [initial_state]
errors = []
v_s = []
for i in range(2500):
    x_current, _, _ = p_ctrl_plant.current_state
    e = x_target - x_current
    errors.append(e)
    v_needed = kp * e

    omega_needed = v_needed / r
    new_state = p_ctrl_plant.simulation_step(omega_needed, omega_needed, dt)
    v_s.append(p_ctrl_plant.linear_velocity)


    state_history.append(new_state)

plot_trajectory(state_history)
plot_errors(errors, dt)
plot_velocity(v_s, dt)


# Trying to implement PI Controller for the speed

In [ ]:
v_target = 10
dt = 0.01

kp = 0.01
ki = 0.01

err_acc = 0

p_ctrl_plant = DifferentialDriveRobot(r, L, initial_state, omega_cap = 20)

state_history = [initial_state]
errors = []
v_s = []
err_acc_history = []
for i in range(1500):
    _, _, _ = p_ctrl_plant.current_state
    current_velocity = p_ctrl_plant.linear_velocity

    e = v_target - current_velocity
    errors.append(e)
    err_acc += e * dt
    err_acc_history.append(err_acc)

    p_compotent = kp * e
    i_compontent = err_acc * ki

    v_needed = p_compotent + i_compontent

    omega_needed = v_needed / r
    new_state = p_ctrl_plant.simulation_step(omega_needed, omega_needed, dt)
    v_s.append(p_ctrl_plant.linear_velocity)


    state_history.append(new_state)

plot_errors(errors, dt)
plot_velocity(v_s, dt)
plot_errors(err_acc_history, dt)


# Implementing P Controller for the orientation only

In [ ]:
theta_target = 10
dt = 0.01

kp = 0.5

p_ctrl_plant = DifferentialDriveRobot(r, L, initial_state, omega_cap = 20)

state_history = [initial_state]
errors = []
v_s = []
for i in range(1000):
    _, _, theta = p_ctrl_plant.current_state

    e = theta_target - theta
    errors.append(e)

    p_compotent = kp * e
    theta_needed = p_compotent

    omega_needed = theta_needed * L / (2 * r)

    new_state = p_ctrl_plant.simulation_step(-omega_needed, omega_needed, dt)
    v_s.append(p_ctrl_plant.linear_velocity)

    state_history.append(new_state)

plot_errors(errors, dt)
plot_trajectory(state_history)
plot_velocity(v_s, dt)
plot_theta( [theta for _, _, theta in state_history], dt )


# P regulator tracking position and directions

In [ ]:
x_target = 10
y_target = 15

dt = 0.01
k_pos = 0.1
k_theta = 0.5

p_ctrl_plant = DifferentialDriveRobot(r, L, initial_state)

state_history = [initial_state]
errors_theta = []
errors_distance = []

v_s = []
for i in range(5000):
    x, y, theta = p_ctrl_plant.current_state

    # theta
    theta_desired = np.arctan2(y_target - y, x_target - x)
    error_theta = np.arctan2(np.sin(theta_desired - theta), np.cos(theta_desired - theta))
    errors_theta.append(error_theta)

    p_compotent = k_theta * error_theta
    omega_needed = p_compotent

    # position
    distance_error = np.sqrt((x_target - x) ** 2 + (y_target - y) ** 2)
    v_needed = k_pos * distance_error
    errors_distance.append(distance_error)

    # omegas
    v_r = v_needed + L*omega_needed/2
    v_l = v_needed - L*omega_needed/2

    omega_r_needed = v_r / r
    omega_l_needed = v_l / r

    new_state = p_ctrl_plant.simulation_step(omega_l_needed, omega_r_needed, dt)
    v_s.append(p_ctrl_plant.linear_velocity)

    state_history.append(new_state)

plot_errors(errors_distance, dt, "Distance error")
plot_errors(errors_theta, dt, "Angle error")
plot_trajectory(state_history)
plot_velocity(v_s, dt)
plot_theta( [theta for _, _, theta in state_history], dt )


In [ ]:
x_target = 10
y_target = 15

waypoints = [
    (10, 15),
    (0, 0),
    (-10, 15),
    (0, 0),
    (-10, -15),
    (0,0),
    (10, -15),
    (0,0),
    (0.0, 0.0),
    (6.0, 0.0),
    (10.0, 0.0),
    (10.0, 4.0),
    (6.0, 4.0),
    (6.0, 8.0),
    (10.0, 8.0),
    (10.0, 12.0),
    (6.0, 12.0),
    (2.0, 12.0),
    (2.0, 8.0),
    (-2.0, 8.0),
    (-2.0, 4.0),
    (2.0, 4.0),
    (2.0, 0.0),
    (-2.0, 0.0),
    (-6.0, 0.0),
    (-6.0, -4.0),
    (-2.0, -4.0),
    (-2.0, -8.0),
    (2.0, -8.0),
    (6.0, -8.0),
    (6.0, -4.0),
    (10.0, -4.0),
    (10.0, -10.0),
    (6.0, -10.0),
    (2.0, -10.0),
    (0.0, -6.0)
    # (10, 10),
    # (0, 0)
]

dt = 0.01
k_pos = 0.2
k_theta = 1.0
target_radius = 0.5

p_ctrl_plant = DifferentialDriveRobot(r, L, initial_state)

state_history = [initial_state]
errors_theta = []
errors_distance = []
v_s = []

target_idx = 0

for i in range(25000):
    x, y, theta = p_ctrl_plant.current_state

    wx, wy = waypoints[target_idx]
    dx = wx - x
    dy = wy - y

    distance_error = np.hypot(dx, dy)
    if distance_error < target_radius and target_idx < len(waypoints) - 1:
        target_idx += 1

    theta_desired = np.arctan2(dy, dx)
    error_theta = np.arctan2(np.sin(theta_desired - theta), np.cos(theta_desired - theta))

    v_needed = k_pos * distance_error
    omega_needed = k_theta * error_theta

    v_r = v_needed + (L * omega_needed) / 2
    v_l = v_needed - (L * omega_needed) / 2

    omega_r_needed = v_r / r
    omega_l_needed = v_l / r

    new_state = p_ctrl_plant.simulation_step(omega_l_needed, omega_r_needed, dt)

    errors_theta.append(error_theta)
    errors_distance.append(distance_error)
    v_s.append(v_needed)
    state_history.append(new_state)

plot_errors(errors_distance, dt, "Distance to waypoint")
plot_errors(errors_theta, dt, "Heading error")
plot_trajectory(state_history)
plot_velocity(v_s, dt)
plot_theta([theta for _, _, theta in state_history], dt)

# Introducing f(theta_error) scaller to make the robot spin in place with theta is too large

In [ ]:
x_target = 10
y_target = 15

waypoints = [
    (10, 15),
    (0, 0),
    (-10, 15),
    (0, 0),
    (-10, -15),
    (0,0),
    (10, -15),
    (0,0),
    (0.0, 0.0),
    (6.0, 0.0),
    (10.0, 0.0),
    (10.0, 4.0),
    (6.0, 4.0),
    (6.0, 8.0),
    (10.0, 8.0),
    (10.0, 12.0),
    (6.0, 12.0),
    (2.0, 12.0),
    (2.0, 8.0),
    (-2.0, 8.0),
    (-2.0, 4.0),
    (2.0, 4.0),
    (2.0, 0.0),
    (-2.0, 0.0),
    (-6.0, 0.0),
    (-6.0, -4.0),
    (-2.0, -4.0),
    (-2.0, -8.0),
    (2.0, -8.0),
    (6.0, -8.0),
    (6.0, -4.0),
    (10.0, -4.0),
    (10.0, -10.0),
    (6.0, -10.0),
    (2.0, -10.0),
    (0.0, -6.0)
]

dt = 0.01
k_pos = 0.2
k_theta = 1.0
target_radius = 0.5

p_ctrl_plant = DifferentialDriveRobot(r, L, initial_state)

state_history = [initial_state]
errors_theta = []
errors_distance = []
v_s = []

target_idx = 0

for i in range(25000):
    x, y, theta = p_ctrl_plant.current_state

    wx, wy = waypoints[target_idx]
    dx = wx - x
    dy = wy - y

    distance_error = np.hypot(dx, dy)
    if distance_error < target_radius and target_idx < len(waypoints) - 1:
        target_idx += 1

    theta_desired = np.arctan2(dy, dx)
    error_theta = np.arctan2(np.sin(theta_desired - theta), np.cos(theta_desired - theta))

    v_scaller = np.cos(np.abs(error_theta)) if np.abs(error_theta) < np.pi / 2 else 0.0

    v_needed = k_pos * distance_error * v_scaller
    omega_needed = k_theta * error_theta

    v_r = v_needed + (L * omega_needed) / 2
    v_l = v_needed - (L * omega_needed) / 2

    omega_r_needed = v_r / r
    omega_l_needed = v_l / r

    new_state = p_ctrl_plant.simulation_step(omega_l_needed, omega_r_needed, dt)

    errors_theta.append(error_theta)
    errors_distance.append(distance_error)
    v_s.append(v_needed)
    state_history.append(new_state)

plot_errors(errors_distance, dt, "Distance to waypoint")
plot_errors(errors_theta, dt, "Heading error")
plot_trajectory(state_history)
plot_velocity(v_s, dt)
plot_theta([theta for _, _, theta in state_history], dt)

# Limitation of current mode

In [ ]:
x_target = 10
y_target = 15

waypoints = [
    (2, 1),
    (4, 0)
]

dt = 0.01
k_pos = 0.2
k_theta = 1.0
target_radius = 0.05

p_ctrl_plant = DifferentialDriveRobot(r, L, initial_state)

state_history = [initial_state]
errors_theta = []
errors_distance = []
v_s = []

target_idx = 0

for i in range(25000):
    x, y, theta = p_ctrl_plant.current_state

    wx, wy = waypoints[target_idx]
    dx = wx - x
    dy = wy - y

    distance_error = np.hypot(dx, dy)
    if distance_error < target_radius and target_idx < len(waypoints) - 1:
        target_idx += 1

    theta_desired = np.arctan2(dy, dx)
    error_theta = np.arctan2(np.sin(theta_desired - theta), np.cos(theta_desired - theta))

    v_scaller = np.cos(np.abs(error_theta)) if np.abs(error_theta) < np.pi / 2 else 0.0

    v_needed = k_pos * distance_error * v_scaller
    omega_needed = k_theta * error_theta

    v_r = v_needed + (L * omega_needed) / 2
    v_l = v_needed - (L * omega_needed) / 2

    omega_r_needed = v_r / r
    omega_l_needed = v_l / r

    new_state = p_ctrl_plant.simulation_step(omega_l_needed, omega_r_needed, dt)

    errors_theta.append(error_theta)
    errors_distance.append(distance_error)
    v_s.append(v_needed)
    state_history.append(new_state)

plot_errors(errors_distance, dt, "Distance to waypoint")
plot_errors(errors_theta, dt, "Heading error")
plot_trajectory(state_history)
plot_velocity(v_s, dt)
plot_theta([theta for _, _, theta in state_history], dt)

# Path follower

In [ ]:
def path(s):
    x = s
    y = np.sin(s)
    return x, y

def find_closest_point_on_path(x, y, path_func, num_samples=100):
    s_start = x - 15
    s_end = x + 15

    s_values = np.linspace(s_start, s_end, num_samples)
    closest_s = None
    closest_distance = float('inf')

    for s in s_values:
        path_x, path_y = path_func(s)
        distance = np.sqrt((x - path_x) ** 2 + (y - path_y) ** 2)

        if distance < closest_distance:
            closest_distance = distance
            closest_s = s

    return closest_s

def find_path_tangent_angle(s, path_func):
    delta = 1e-5
    x1, y1 = path_func(s)
    x2, y2 = path_func(s + delta)

    dx = x2 - x1
    dy = y2 - y1

    angle = np.arctan2(dy, dx)
    return angle

def calc_cross_track_error(x, y, path_func, s):
    path_x, path_y = path_func(s)

    # Calculate the tangent angle at the closest point
    tangent_angle = find_path_tangent_angle(s, path_func)

    # Vector from the closest point on the path to the robot's position
    vector_to_robot = np.array([x - path_x, y - path_y])

    # Normal vector to the path (perpendicular to the tangent)
    normal_vector = np.array([-np.sin(tangent_angle), np.cos(tangent_angle)])

    # Cross-track error is the projection of the vector_to_robot onto the normal_vector
    cross_track_error = np.dot(vector_to_robot, normal_vector)

    return cross_track_error

initial_state = (5, 5, -1)

dt = 0.01

k_e = 1.0
k_theta = 1.0
v_path = 0.5

p_ctrl_plant = DifferentialDriveRobot(r, L, initial_state)

state_history = [initial_state]
errors_theta = []
errors_cross_track = []
v_s = []

for i in range(5000):

    # Robot state
    x, y, theta = p_ctrl_plant.current_state

    # Find closest point on path
    closest_s = find_closest_point_on_path(
        x, y, path
    )

    # Path direction at closest point
    path_tangent_angle = find_path_tangent_angle(
        closest_s, path
    )

    # Signed cross-track error
    cross_track_error = calc_cross_track_error(
        x, y, path, closest_s
    )

    # Desired heading
    theta_desired = (
        path_tangent_angle
        + np.arctan(-k_e * cross_track_error)
    )

    # Wrapped heading error
    error_theta = np.arctan2(
        np.sin(theta_desired - theta),
        np.cos(theta_desired - theta)
    )

    # Heading controller
    omega_needed = k_theta * error_theta

    # Constant forward velocity
    v_needed = v_path

    # Differential-drive inverse kinematics
    v_r = v_needed + (L * omega_needed) / 2
    v_l = v_needed - (L * omega_needed) / 2

    omega_r_needed = v_r / r
    omega_l_needed = v_l / r

    # Apply command
    new_state = p_ctrl_plant.simulation_step(
        omega_l_needed,
        omega_r_needed,
        dt
    )

    # Store data
    errors_theta.append(error_theta)
    errors_cross_track.append(cross_track_error)
    v_s.append(v_needed)
    state_history.append(new_state)


# Results
plot_errors(
    errors_cross_track,
    dt,
    "Cross-track error"
)

plot_errors(
    errors_theta,
    dt,
    "Heading error"
)

plot_trajectory(state_history)

plot_velocity(v_s, dt)

plot_theta(
    [theta for _, _, theta in state_history],
    dt
)

## Complex Path Examples

In [ ]:
# Spiral path - expands outward
def path_spiral(s):
    """Logarithmic spiral - starts tight, expands outward"""
    radius = 0.5 * np.exp(s / 10)  # Exponential growth
    x = radius * np.cos(s)
    y = radius * np.sin(s)
    return x, y

# Figure-8 path (lemniscate)
def path_figure8(s):
    """Figure-8 curve - crosses itself"""
    x = 3 * np.sin(s)
    y = 3 * np.sin(s) * np.cos(s)
    return x, y

# S-curve path
def path_s_curve(s):
    """Smooth S-shaped path"""
    x = s
    y = 2 * np.tanh(2 * s)  # Smooth sigmoid-like curve
    return x, y

# Circular path
def path_circular(s):
    """Perfect circle with radius 10"""
    radius = 10
    x = radius * np.cos(s / radius)
    y = radius * np.sin(s / radius)
    return x, y

# Rose curve (polar-based)
def path_rose(s):
    """Rose curve - beautiful flower-like pattern"""
    k = 5  # Number of petals
    r = 2.5 * np.cos(k * s)
    x = r * np.cos(s)
    y = r * np.sin(s)
    return x, y

# Bezier-like smooth path
def path_smooth_wave(s):
    """Smooth wave with varying frequency"""
    x = s
    y = 2 * np.sin(s) + 0.8 * np.cos(2 * s) + 0.4 * np.sin(3 * s)
    return x, y

# Square spiral
def path_square_spiral(s):
    """Spiral that traces a square pattern"""
    amplitude = s / 10
    frequency = s
    x = amplitude * np.cos(frequency)
    y = amplitude * np.sin(frequency) + 0.2 * s
    return x, y

In [ ]:
# Visualize all paths
paths = {
    "Simple Sine": path,
    "Spiral": path_spiral,
    "Figure-8": path_figure8,
    "S-Curve": path_s_curve,
    "Circle": path_circular,
    "Rose": path_rose,
    "Smooth Wave": path_smooth_wave,
    "Square Spiral": path_square_spiral,
}

fig = go.Figure()

for name, path_func in paths.items():
    if name == "Spiral":
        s_vals = np.linspace(0, 6, 500)
    elif name == "Square Spiral":
        s_vals = np.linspace(0, 10, 500)
    else:
        s_vals = np.linspace(0, 4 * np.pi, 500)

    x_vals = [path_func(s)[0] for s in s_vals]
    y_vals = [path_func(s)[1] for s in s_vals]

    fig.add_trace(go.Scatter(
        x=x_vals, y=y_vals,
        mode='lines',
        name=name,
        visible=True
    ))

fig.update_layout(
    title='Different Path Shapes for Following',
    xaxis_title='X',
    yaxis_title='Y',
    hovermode='closest',
    height=600
)

fig.show()

## Test All Paths with Path Follower

Run the path following controller on each complex path and visualize results.

In [ ]:
# Path following parameters
dt = 0.01
k_e = 5
k_theta = 5
v_path = 0.5
initial_state_pf = (0.5, 0.5, 0)

# Dictionary of all paths to test
paths_to_test = {
    "Sine Wave": path,
    "Spiral": path_spiral,
    "Figure-8": path_figure8,
    "S-Curve": path_s_curve,
    "Circle": path_circular,
    "Rose": path_rose,
    "Smooth Wave": path_smooth_wave,
    "Square Spiral": path_square_spiral,
}

# Run experiment for each path
results = {}

for path_name, path_func in paths_to_test.items():
    print(f"\n{'='*50}")
    print(f"Testing: {path_name}")
    print(f"{'='*50}")

    n_steps = 5000

    # Initialize robot
    p_ctrl_plant = DifferentialDriveRobot(r, L, initial_state_pf)

    # Storage
    state_history = [initial_state_pf]
    errors_cross_track = []
    errors_theta = []

    # Simulation
    for i in range(n_steps):
        x, y, theta = p_ctrl_plant.current_state

        # Find closest point on path
        closest_s = find_closest_point_on_path(x, y, path_func)

        # Path direction at closest point
        path_tangent_angle = find_path_tangent_angle(closest_s, path_func)

        # Signed cross-track error
        cross_track_error = calc_cross_track_error(x, y, path_func, closest_s)

        # Desired heading
        theta_desired = (
            path_tangent_angle
            + np.arctan(-k_e * cross_track_error)
        )

        # Wrapped heading error
        error_theta = np.arctan2(
            np.sin(theta_desired - theta),
            np.cos(theta_desired - theta)
        )

        # Heading controller
        omega_needed = k_theta * error_theta

        # Forward velocity
        v_needed = v_path

        # Differential-drive inverse kinematics
        v_r = v_needed + (L * omega_needed) / 2
        v_l = v_needed - (L * omega_needed) / 2

        omega_r_needed = v_r / r
        omega_l_needed = v_l / r

        # Apply command
        new_state = p_ctrl_plant.simulation_step(
            omega_l_needed,
            omega_r_needed,
            dt
        )

        # Store data
        errors_cross_track.append(cross_track_error)
        errors_theta.append(error_theta)
        state_history.append(new_state)

    # Store results
    results[path_name] = {
        'state_history': state_history,
        'errors_cross_track': errors_cross_track,
        'errors_theta': errors_theta,
        'path_func': path_func
    }

    print(f"Final cross-track error: {errors_cross_track[-1]:.4f}")
    print(f"Mean cross-track error: {np.mean(np.abs(errors_cross_track)):.4f}")
    print(f"Max cross-track error: {np.max(np.abs(errors_cross_track)):.4f}")

print(f"\n{'='*50}")
print("All experiments completed!")
print(f"{'='*50}")


Testing: Sine Wave
Final cross-track error: 0.0408
Mean cross-track error: 0.0154
Max cross-track error: 0.0469

Testing: Spiral
Final cross-track error: -0.0168
Mean cross-track error: 0.0842
Max cross-track error: 1.1463

Testing: Figure-8
Final cross-track error: 0.0150
Mean cross-track error: 0.0136
Max cross-track error: 0.0983

Testing: S-Curve
Final cross-track error: -0.0000
Mean cross-track error: 0.0074
Max cross-track error: 0.4017

Testing: Circle
Final cross-track error: 0.1258
Mean cross-track error: 1.7685
Max cross-track error: 9.2929

Testing: Rose
Final cross-track error: -0.0223
Mean cross-track error: 0.0277
Max cross-track error: 0.2565

Testing: Smooth Wave
Final cross-track error: -0.0178
Mean cross-track error: 0.0481
Max cross-track error: 0.6123

Testing: Square Spiral
Final cross-track error: -0.0248
Mean cross-track error: 0.0241
Max cross-track error: 0.2459

All experiments completed!


In [ ]:
# Helper function to plot path + robot trajectory
def plot_path_and_trajectory(path_name, path_func, state_history, errors_cross_track):
    """Plot reference path and robot trajectory together"""

    # Generate reference path
    if path_name == "Spiral":
        s_vals = np.linspace(0, 6, 300)
    elif path_name == "Square Spiral":
        s_vals = np.linspace(0, 10, 300)
    else:
        s_vals = np.linspace(0, 4 * np.pi, 300)

    path_x = [path_func(s)[0] for s in s_vals]
    path_y = [path_func(s)[1] for s in s_vals]

    # Robot trajectory
    robot_x = [pos[0] for pos in state_history]
    robot_y = [pos[1] for pos in state_history]

    # Create figure
    fig = go.Figure()

    # Add reference path
    fig.add_trace(go.Scatter(
        x=path_x,
        y=path_y,
        mode='lines',
        name='Reference Path',
        line={"color": 'blue', "width": 3, "dash": 'dash'},
        hovertemplate='Path<br>X: %{x:.3f}<br>Y: %{y:.3f}<extra></extra>'
    ))

    # Add robot trajectory
    fig.add_trace(go.Scatter(
        x=robot_x,
        y=robot_y,
        mode='lines+markers',
        name='Robot Trajectory',
        line={"color": 'red', "width": 2},
        marker={"size": 3},
        hovertemplate='Robot<br>X: %{x:.3f}<br>Y: %{y:.3f}<extra></extra>'
    ))

    # Add start point
    fig.add_trace(go.Scatter(
        x=[robot_x[0]],
        y=[robot_y[0]],
        mode='markers',
        name='Start',
        marker={"color": 'green', "size": 12, "symbol": 'circle'}
    ))

    # Add end point
    fig.add_trace(go.Scatter(
        x=[robot_x[-1]],
        y=[robot_y[-1]],
        mode='markers',
        name='End',
        marker={"color": 'orange', "size": 12, "symbol": 'square'}
    ))

    # Stats
    mean_error = np.mean(np.abs(errors_cross_track))
    max_error = np.max(np.abs(errors_cross_track))

    fig.update_layout(
        title=f"{path_name} - Path Following<br><sub>Mean Error: {mean_error:.4f} m, Max Error: {max_error:.4f} m</sub>",
        xaxis_title='X Position (m)',
        yaxis_title='Y Position (m)',
        hovermode='closest',
        showlegend=True,
        height=600,
        width=800
    )

    # Equal aspect ratio
    fig.update_yaxes(scaleanchor="x", scaleratio=1)

    return fig

# Plot all results
for path_name, result in results.items():
    fig = plot_path_and_trajectory(
        path_name,
        result['path_func'],
        result['state_history'],
        result['errors_cross_track']
    )
    fig.show()

In [ ]:
# Performance comparison table
import pandas as pd

performance_data = []

for path_name, result in results.items():
    errors = np.array(result['errors_cross_track'])
    theta_errors = np.array(result['errors_theta'])

    performance_data.append({
        'Path': path_name,
        'Mean CTE (m)': f"{np.mean(np.abs(errors)):.4f}",
        'Max CTE (m)': f"{np.max(np.abs(errors)):.4f}",
        'Std CTE (m)': f"{np.std(errors):.4f}",
        'Mean Theta Error (rad)': f"{np.mean(np.abs(theta_errors)):.4f}",
        'Max Theta Error (rad)': f"{np.max(np.abs(theta_errors)):.4f}",
    })

df = pd.DataFrame(performance_data)
print("\n" + "="*120)
print("PATH FOLLOWING PERFORMANCE SUMMARY")
print("="*120)
print(df.to_string(index=False))
print("="*120)
print("\nCTE = Cross-Track Error")


PATH FOLLOWING PERFORMANCE SUMMARY
         Path Mean CTE (m) Max CTE (m) Std CTE (m) Mean Theta Error (rad) Max Theta Error (rad)
    Sine Wave       0.0154      0.0469      0.0201                 0.0520                0.6167
       Spiral       0.0842      1.1463      0.2217                 0.1196                3.0944
     Figure-8       0.0136      0.0983      0.0144                 0.0790                1.3033
      S-Curve       0.0074      0.4017      0.0442                 0.0174                2.3909
       Circle       1.7685      9.2929      2.7642                 0.0187                0.8139
         Rose       0.0277      0.2565      0.0436                 0.3088                3.1185
  Smooth Wave       0.0481      0.6123      0.0880                 0.1228                2.4864
Square Spiral       0.0241      0.2459      0.0273                 0.1374                1.6866

CTE = Cross-Track Error
